# Analisis conjunto - Campeche

> La composicion de las unidades economicas de Campeche, se corresponde con el comportamiento de su actividad economica agregada?

**Division de trabajo ("por partes"):**
- Seccion 1 (consulta JOIN): **Aremy**
- Seccion 2 (grafica): **Abigail**
- Secciones 3 y 4 (conclusion y limitaciones): las dos

**Estado (20/ago/2026):** ambas capas cargadas y verificadas: BIE (4 series, 2022-Q1 a 2026-Q1) y DENUE (47,821 establecimientos). Si tienes que reejecutar, corre primero `python main.py` en esta rama y luego el notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd()
if RAIZ.name == 'integracion':
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from core.db import consultar

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Seccion 1 · Consulta con JOIN (Aremy)

Puente: `denue_establecimiento -> dim_sector_actividad (sector_id) -> gran_division <- -> bie_indicador.gran_division -> bie_observacion (indicador_id)`.

Dos advertencias que hay que tener presentes:
1. **`Total` no aparece**: `dim_sector_actividad` solo tiene `Primarias/Secundarias/Terciarias`, asi que la serie `Total` del ITAEE no tiene contraparte micro. Queda fuera por diseno.
2. **Micro = fotografia, macro = serie de tiempo**: el conteo de establecimientos es un numero fijo que se repite en cada trimestre al hacer el JOIN.

In [ ]:
SQL_JOIN = '''
WITH micro AS (
    SELECT d.gran_division,
           COUNT(*) AS n_establecimientos
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
),
macro AS (
    SELECT bi.gran_division,
           o.anio,
           o.trimestre,
           o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
)
SELECT mi.gran_division,
       mi.n_establecimientos,
       ma.anio,
       ma.trimestre,
       ma.valor
FROM micro mi
JOIN macro ma ON mi.gran_division = ma.gran_division
ORDER BY mi.gran_division, ma.anio, ma.trimestre;
'''

df = consultar(SQL_JOIN)
print('Filas del JOIN:', len(df))
df.head()

In [ ]:
# Variante: conteo vs valor del ITAEE en el trimestre mas reciente.
# Evita repetir el conteo en cada trimestre y da el cuadro "aqui y ahora".
SQL_PUNTO = '''
WITH micro AS (
    SELECT d.gran_division, COUNT(*) AS n_establecimientos
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
),
ultimo AS (
    SELECT bi.gran_division, o.periodo
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
      AND o.periodo = (SELECT MAX(o2.periodo) FROM bie_observacion o2
                       WHERE o2.area_geografica = '04'
                         AND o2.indicador_id = o.indicador_id)
),
macro AS (
    SELECT bi.gran_division, o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    JOIN ultimo u ON u.gran_division = bi.gran_division
                 AND u.periodo = o.periodo
    WHERE o.area_geografica = '04'
)
SELECT m.gran_division,
       m.n_establecimientos,
       ROUND(m.n_establecimientos * 100.0 / SUM(m.n_establecimientos) OVER (), 1) AS pct_establecimientos,
       ROUND(ma.valor, 2) AS itaee_ultimo_trimestre
FROM micro m
JOIN macro ma ON m.gran_division = ma.gran_division;
'''

df_punto = consultar(SQL_PUNTO)
print('Cuadro final (micro vs macro, ultimo trimestre):')
df_punto

## Seccion 2 · Grafica (Abigail)

Los conteos (micro) y los indices (macro) son magnitudes incomparables. La solucion elegida:
- Panel izquierdo: **participacion (%)** de los establecimientos por gran division (composicion micro).
- Panel derecho: serie del **ITAEE** (indice base 2018=100) por gran division (comportamiento macro).

Asi se leen las dos cosas sin fingir que comparten escala.

In [ ]:
# Grafica (Abigail): participacion micro vs serie macro.
micro = consultar('''
    SELECT d.gran_division, COUNT(*) AS n
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
''')

macro = consultar('''
    SELECT bi.gran_division, o.anio, o.trimestre, o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
''')
macro['periodo'] = macro['anio'].astype(str) + '-Q' + macro['trimestre'].astype(str)

orden = ['Primarias', 'Secundarias', 'Terciarias']
colores = ['#2ca02c', '#d62728', '#1f77b4']

fig, (ax_izq, ax_der) = plt.subplots(1, 2, figsize=(12, 4.5))

if micro.empty:
    ax_izq.text(0.5, 0.5, 'Sin datos micro (DENUE no cargado).\nReejecutar `python main.py`.',
                ha='center', va='center', transform=ax_izq.transAxes)
else:
    micro_pct = (micro.set_index('gran_division').loc[orden]['n'] / micro['n'].sum() * 100)
    barras = ax_izq.bar(micro_pct.index, micro_pct, color=colores)
    ax_izq.bar_label(barras, fmt='%.1f%%')
    ax_izq.set_ylabel('Participacion en establecimientos (%)')
    ax_izq.set_title('Composicion micro (DENUE)')
    ax_izq.set_ylim(0, 100)

for gd, color in zip(orden, colores):
    s = macro[macro['gran_division'] == gd].sort_values(['anio', 'trimestre'])
    ax_der.plot(s['periodo'], s['valor'], marker='o', markersize=3, label=gd, color=color)
ax_der.axhline(100, color='gray', lw=0.8, ls='--', alpha=0.6)
ax_der.set_ylabel('ITAEE (base 2018 = 100)')
ax_der.set_title('Actividad macro (BIE) - Campeche')
ax_der.legend(fontsize=8)
ax_der.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Numeros clave reproducibles para la conclusion.
if not micro.empty:
    total = micro['n'].sum()
    print(f"Establecimientos totales: {total:,}")
    print(micro.assign(pct=round(micro['n'] / total * 100, 1)).to_string(index=False))
else:
    print('Micro vacio (DENUE no cargado aun).')

print()
print('ITAEE ultimo trimestre (2026-Q1):')
print(consultar('''
    SELECT bi.gran_division, ROUND(o.valor, 2) AS valor
    FROM bie_observacion o JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04' AND o.anio = 2026 AND o.trimestre = 1
''').to_string(index=False))

## Seccion 3 · Conclusion (media cuartilla, las dos)

**Aporte de Abigail** (Aremy lo revisa y ajusta):

No se corresponden. La composición de las unidades económicas de Campeche está dominada por las actividades terciarias, mientras que el comportamiento de su actividad agregada lo dictan otras. Esa diferencia es el hallazgo, no un error de los datos.

Del lado micro, 47,821 establecimientos forman una economía de negocios pequeños: 87% son terciarios (41,706), encabezados por el comercio al por menor (18,916), otros servicios (7,213) y alimentos y hospedaje (6,756). Las actividades primarias apenas representan 2.7% del censo y las secundarias 10.1%.

Del lado macro, el ITAEE (base 2018 = 100) cuenta otra historia. Las terciarias se sostienen como un piso estable alrededor del nivel base: los miles de comercios sostienen actividad, pero no la impulsan. Las secundarias, que tienen solo el 10% de los establecimientos, cayeron cerca de 20% entre 2022 y 2026 y se ubican más de 30 puntos por debajo de su base: un movimiento de ese tamaño con tan pocas unidades solo ocurre en sectores concentrados con gran peso en el producto. Las primarias, pocas pero volátiles, oscilan con la estacionalidad agropecuaria.

En una frase: en Campeche los negocios que se cuentan por miles no son los que mueven el producto. Medir la economía por el número de establecimientos diría que Campeche es una economía de servicios; medirla por su actividad agregada muestra que la dinámica la marcan unos cuantos sectores con pocas unidades y mucho peso.

## Seccion 4 · Limitaciones (las dos)

**Borrador ampliado por Abigail** (Aremy edita):

- El DENUE cuenta **establecimientos**, no empleo ni produccion. Una cadena con 40 sucursales aparece 40 veces, y los negocios sin registro pueden no estar en el directorio.
- El ITAEE es un **indice** (base 2018=100), no un monto: no se puede sumar entre divisiones ni afirmar cuanto "pesa" cada sector en el producto; solo se compara la trayectoria.
- **Fotografia vs serie**: el DENUE es un corte en el tiempo; el ITAEE, una serie. Comparar su participacion presupone que la estructura actual de negocios es representativa de todo el periodo, y hay desfase: el censo es de 2026 y la serie arranca en 2022.
- **Estacionalidad de las primarias**: un solo trimestre como "nivel" seria enganoso; por eso la conclusion usa trayectorias, no niveles puntuales.
- No se puede concluir causalidad ni que "los comercios producen poco": para eso haria falta valor agregado por tamano o estrato.
- El conteo incluye todos los tamanos (de micro a grande); sin el estrato como llave no sabemos cuanto empleo absorbe cada sector.
- El `Total` del ITAEE queda fuera del JOIN porque el catalogo de sectores no tiene una categoria "Total".
- La comparacion depende de que ambas fuentes compartan el clasificador SCIAN 2018; un cambio de clasificador en cualquiera de las dos romperia el puente en silencio.